In [22]:
import os

In [23]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\mlProject'

In [5]:
os.chdir("../")


'c:\\Users\\Lenovo\\Desktop\\mlProject'

In [33]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\mlProject'

In [34]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

In [35]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [36]:
from stb_pfe_mlflow.constants import *
from stb_pfe_mlflow.utils.common import read_yaml, create_directories

In [37]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

        return data_transformation_config

In [ ]:
import os
from stb_pfe_mlflow import logger
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler



In [40]:
df= pd.read_csv("artifacts/data_cleaning/clean_data.csv")
df.head()

,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,peda_edema,aanemia,class
0,48.0,80.0,1.020,1.0,0.0,normal,normal,notpresent,notpresent,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,normal,normal,notpresent,notpresent,121.0,...,38,6000,5.2,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31,7500,5.2,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [41]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        
        
        
    def transforming_data(self):
        
        cat_cols = [col for col in df.columns if df[col].dtype == 'object']
        num_cols = [col for col in df.columns if df[col].dtype != 'object']
        le = LabelEncoder()

        for col in cat_cols:
            df[col] = le.fit_transform(df[col])
        
        scaler = StandardScaler()
        for col in num_cols:
            df[col]= scaler.fit_transform(df[num_cols])
    
    
        # Enregistrer le dataset final dans le répertoire configuré
        df.to_csv(os.path.join(self.config.root_dir, "transforming_data.csv"), index=False)

        # Log information
        logger.info("Data transformation complete")
        logger.info(f"Data shape: {df.shape}")

In [42]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transforming_data()
except Exception as e:
    raise e

[2024-11-20 22:08:55,397: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-11-20 22:08:55,398: INFO: common: yaml file: params.yaml loaded successfully]
[2024-11-20 22:08:55,401: INFO: common: yaml file: schema.yaml loaded successfully]
[2024-11-20 22:08:55,402: INFO: common: created directory at: artifacts]
[2024-11-20 22:08:55,404: INFO: common: created directory at: artifacts/data_transformation]
[2024-11-20 22:08:55,445: INFO: 923752688: Data transformation complete]
[2024-11-20 22:08:55,446: INFO: 923752688: Data shape: (400, 25)]


In [43]:
df

,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,peda_edema,aanemia,class
0,-0.210031,-0.210031,-0.210031,-0.210031,-0.210031,1,1,0,0,-0.210031,...,32,72,34,1,1,0,0,0,0,0
1,-2.627234,-2.627234,-2.627234,-2.627234,-2.627234,1,1,0,0,-2.627234,...,26,56,34,0,0,0,0,0,0,0
2,0.615355,0.615355,0.615355,0.615355,0.615355,1,1,0,0,0.615355,...,19,70,34,0,1,0,1,0,1,0
3,-0.210031,-0.210031,-0.210031,-0.210031,-0.210031,1,0,1,0,-0.210031,...,20,62,19,1,0,0,1,1,1,0
4,-0.033163,-0.033163,-0.033163,-0.033163,-0.033163,1,1,0,0,-0.033163,...,23,68,27,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,0.202662,0.202662,0.202662,0.202662,0.202662,1,1,0,0,0.202662,...,35,62,30,0,0,0,0,0,0,1
396,-0.563768,-0.563768,-0.563768,-0.563768,-0.563768,1,1,0,0,-0.563768,...,42,72,44,0,0,0,0,0,0,1
397,-2.332453,-2.332453,-2.332453,-2.332453,-2.332453,1,1,0,0,-2.332453,...,37,61,36,0,0,0,0,0,0,1
398,-2.037673,-2.037673,-2.037673,-2.037673,-2.037673,1,1,0,0,-2.037673,...,39,67,41,0,0,0,0,0,0,1
